# FastMCP 로컬 서버 제작

일반 Python 함수를 MCP 클라이언트가 발견하고 호출할 수 있는 서버 구성 요소로 만들려면 무엇을 등록해야 할까요?

## 실행 환경 준비

Python 3.12 새 커널에서 첫 셀부터 순서대로 실행합니다. 비밀 값은 출력하지 않습니다.

## 패키지 버전 확인

패키지 버전 확인에 필요한 값과 결과를 확인합니다.

In [1]:
# 패키지 버전 확인에 필요한 값을 준비하고 결과를 확인합니다.
%pip install -q "langchain[mcp]==1.4.0" langchain-openai==1.6.0 fastmcp==4.0.3 mcp==2.2.0 python-dotenv==1.2.3
import warnings
from importlib.metadata import version
from langchain_core._api import LangChainBetaWarning

warnings.filterwarnings("ignore", message=r".*langchain\.mcp.*", category=LangChainBetaWarning)

for package in ["langchain", "langchain-openai", "fastmcp", "mcp"]:
    print(package, version(package))

Note: you may need to restart the kernel to use updated packages.


langchain 1.4.0
langchain-openai 1.6.0
fastmcp 4.0.3
mcp 2.2.0


## Before: 등록하지 않은 함수

Before: 등록하지 않은 함수에 필요한 값과 결과를 확인합니다.

In [2]:
# Before: 등록하지 않은 함수에 필요한 값을 준비하고 결과를 확인합니다.
import asyncio
from fastmcp import Client, FastMCP
from fastmcp.exceptions import ToolError

before = FastMCP("등록 전 서버")

def lookup_policy(topic: str) -> str:
    return {"대출": "14일"}[topic]

async def run_before() -> tuple[list[str], str]:
    async with Client(before) as client:
        names = [item.name for item in await client.list_tools()]
        try:
            await client.call_tool("lookup_policy", {"topic": "대출"})
        except ToolError as error:
            return names, type(error).__name__

before_names, before_error = await run_before()
print("tools:", before_names)
print("error:", before_error)
assert before_names == []

tools: []
error: ToolError


## After: 도구 등록

After: 도구 등록에 필요한 값과 결과를 확인합니다.

In [3]:
# After: 도구 등록에 필요한 값을 준비하고 결과를 확인합니다.
after = FastMCP("등록 후 서버")

@after.tool
def lookup_policy(topic: str) -> str:
    """자료실 규정을 조회합니다. topic은 대출입니다."""
    return {"대출": "일반 도서는 14일 동안 대출할 수 있습니다."}[topic]

async def run_after() -> tuple[list[str], str]:
    async with Client(after) as client:
        names = [item.name for item in await client.list_tools()]
        result = await client.call_tool("lookup_policy", {"topic": "대출"})
        return names, result.content[0].text

after_names, after_text = await run_after()
print("tools:", after_names)
print("result:", after_text)
assert after_names == ["lookup_policy"]
assert "14일" in after_text

tools: ['lookup_policy']
result: 일반 도서는 14일 동안 대출할 수 있습니다.


## 완성 서버의 세 구성 요소 확인

완성 서버의 세 구성 요소 확인에 필요한 값과 결과를 확인합니다.

In [4]:
# 완성 서버의 세 구성 요소 확인에 필요한 값을 준비하고 결과를 확인합니다.
import importlib.util
from pathlib import Path

server_path = Path.cwd() / "labs/part6/ch28-mcp/course_server.py"
assert server_path.is_file(), f"서버 파일을 찾지 못했습니다: {server_path}"
spec = importlib.util.spec_from_file_location("course_server", server_path)
course_server = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(course_server)
print("server:", course_server.mcp.name)

async def inspect_server() -> tuple[list[str], list[str], list[str]]:
    async with Client(course_server.mcp) as client:
        return (
            [item.name for item in await client.list_tools()],
            [str(item.uri) for item in await client.list_resources()],
            [item.name for item in await client.list_prompts()],
        )

print(await inspect_server())

server: 교육센터 자료실
(['lookup_policy'], ['policy://handbook'], ['explain_policy'])
